# 🛵 Caso 3 — Rapidinho Delivery: fechas, horarios y dashboard operativo

> **Módulos del MBA relacionados:** Introdução à Programação com Python II–III · Business Intelligence e Data Visualization II–III

## El caso
**Rapidinho** es una app de delivery de comida que opera en 8 barrios de São Paulo. La promesa al cliente (el **SLA**) es entregar en **45 minutos o menos**.

Camila, la gerente de Operaciones, está preocupada: "Las quejas por demora subieron, sobre todo cuando llueve. No sé si me faltan repartidores, en qué horarios ni en qué barrios". Te pasa los datos del primer semestre de 2026 y quiere un **dashboard operativo** para la reunión semanal del equipo.

## Tu misión: responder estas preguntas
1. ¿Cuál es el tiempo medio de entrega y qué % de entregas cumple el SLA? ¿En qué barrios fallamos más?
2. ¿En qué días y horarios se concentran los pedidos? (mapa de calor)
3. ¿Cómo evoluciona la demanda día a día? ¿Hay tendencia?
4. ¿La lluvia aumenta los pedidos? ¿Y las demoras?
5. ¿Las demoras empeoran la nota que deja el cliente?
6. ¿Los cupones aumentan el ticket?

## Diccionario de datos
**entregas.csv** — una fila por pedido

| columna | descripción |
|---|---|
| `pedido_id` | identificador |
| `data_hora_pedido` | fecha y hora del pedido (aaaa-mm-dd hh:mm:ss) |
| `data_hora_entrega` | fecha y hora de la entrega |
| `bairro` | barrio de entrega |
| `categoria_restaurante` | Pizza, Hambúrguer, Japonesa, Brasileira, Saudável, Doces |
| `valor_pedido` | valor de la comida (R\$) |
| `taxa_entrega` | tarifa de envío (R\$) |
| `distancia_km` | distancia restaurante → cliente |
| `entregador_id` | repartidor |
| `cupom` | código de cupón usado (vacío = sin cupón) |
| `avaliacao` | nota 1–5 (vacío si no evaluó) |

**clima.csv** — una fila por día: `data`, `chuva_mm` (lluvia del día), `temp_max_c`

## Qué vas a practicar
`parse_dates` · restar fechas (`timedelta`) · `.dt.hour`, `.dt.dayofweek`, `.dt.normalize()` · `merge` por fecha · `pivot_table` · `rolling` (media móvil) · `resample` · `pd.cut` (franjas) · `quantile` · mapa de calor y dashboard con Plotly

## Paso 0 — Genera la base de datos

Este código simula 6 meses de pedidos y el clima diario, y guarda `entregas.csv` y `clima.csv`. Ejecútalo una vez.

In [ ]:
# ============================================================
# PASO 0 — GENERADOR DE DATOS (ejecútalo una vez)
# ============================================================
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)

# ---------- 1) CLIMA DIARIO ----------
dias = pd.date_range("2026-01-01", "2026-06-30", freq="D")
mes = dias.month.to_numpy()
prob_chuva = np.select([mes <= 2, mes == 3, mes == 4, mes == 5], [0.55, 0.45, 0.3, 0.22], 0.18)
chuva_mm = np.where(rng.random(len(dias)) < prob_chuva, rng.gamma(2, 7, len(dias)).round(1), 0.0)
temp_max = (np.select([mes <= 2, mes == 3, mes == 4, mes == 5], [30.5, 29, 27, 24.5], 23)
            + rng.normal(0, 2, len(dias))).round(1)
clima = pd.DataFrame({"data": dias.strftime("%Y-%m-%d"), "chuva_mm": chuva_mm, "temp_max_c": temp_max})
chuvoso = chuva_mm >= 5

# ---------- 2) ENTREGAS ----------
bairros = pd.DataFrame({
    "bairro": ["Pinheiros", "Vila Mariana", "Moema", "Centro", "Tatuapé", "Santana", "Butantã", "Itaquera"],
    "peso": [16, 15, 13, 12, 13, 11, 10, 10],
    "dist_media": [2.5, 2.8, 3.0, 2.2, 3.5, 3.8, 4.2, 5.5],   # km
    "atraso_extra": [0, 1, 0, 9, 2, 2, 2, 7],                 # tránsito y calles difíciles (min)
})
categorias = ["Pizza", "Hambúrguer", "Japonesa", "Brasileira", "Saudável", "Doces"]
ticket_cat = {"Pizza": 75, "Hambúrguer": 55, "Japonesa": 95, "Brasileira": 45, "Saudável": 50, "Doces": 35}
peso_hora = np.array([1, 0.6, 0.3, 0.1, 0.05, 0.05, 0.1, 0.3, 0.6, 0.8, 1.5, 4,
                      6, 4.5, 2, 1.2, 1.2, 2, 4.5, 7, 7.5, 5, 3, 1.8])

fim_de_semana = dias.dayofweek.to_numpy() >= 4                # viernes a domingo
tendencia = 1 + 0.15 * np.arange(len(dias)) / len(dias)
n_dia = rng.poisson(100 * (1 + 0.3 * fim_de_semana) * (1 + 0.25 * chuvoso) * tendencia)

d = np.repeat(np.arange(len(dias)), n_dia)
n = len(d)
hora = rng.choice(24, size=n, p=peso_hora / peso_hora.sum())
pedido = (dias[d] + pd.to_timedelta(hora, unit="h")
          + pd.to_timedelta(rng.integers(0, 3600, n), unit="s"))
b = rng.choice(len(bairros), size=n, p=bairros["peso"] / bairros["peso"].sum())
dist = rng.gamma(4, bairros["dist_media"].to_numpy()[b] / 4).round(1).clip(0.3)
pico = np.isin(hora, [12, 13, 19, 20])
tempo = (12 + 3.0 * dist + bairros["atraso_extra"].to_numpy()[b] + 7 * pico
         + 11 * chuvoso[d] + rng.gamma(2, 3, n))

cat = rng.choice(categorias, size=n, p=[0.22, 0.2, 0.15, 0.2, 0.1, 0.13])
valor = rng.gamma(6, pd.Series(cat).map(ticket_cat).to_numpy() / 6)
cupom = rng.choice(["", "PRIMEIRA10", "FRETEGRATIS", "VOLTA15"], size=n, p=[0.75, 0.08, 0.1, 0.07])
valor = np.where(cupom != "", valor * 1.12, valor).round(2)  # quien usa cupón pide un poco más
taxa = np.where(cupom == "FRETEGRATIS", 0, (4.9 + 1.2 * dist).round(2))
nota = np.clip(np.round(5.3 - 0.045 * np.maximum(tempo - 25, 0) + rng.normal(0, 0.7, n)), 1, 5)
nota[rng.random(n) < 0.35] = np.nan

entregas = pd.DataFrame({
    "data_hora_pedido": pedido,
    "data_hora_entrega": pedido + pd.to_timedelta(np.round(tempo * 60), unit="s"),
    "bairro": bairros["bairro"].to_numpy()[b],
    "categoria_restaurante": cat,
    "valor_pedido": valor,
    "taxa_entrega": taxa,
    "distancia_km": dist,
    "entregador_id": [f"E{i:03d}" for i in rng.integers(1, 121, n)],
    "cupom": cupom,
    "avaliacao": nota,
}).sort_values("data_hora_pedido").reset_index(drop=True)
entregas.insert(0, "pedido_id", np.arange(1, n + 1))

entregas.to_csv("entregas.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
clima.to_csv("clima.csv", index=False)
print(f"✅ Archivos creados: entregas.csv ({len(entregas)} filas) y clima.csv ({len(clima)} días)")

---
# ✍️ Parte A — Tu turno: el paso a paso en comentarios

Cada celda de abajo tiene **solo comentarios** con lo que debes hacer y algunas pistas.
Escribe tu código debajo de `# ✍️ Tu código aquí:` y ejecútalo.

- Si no conoces una función, búscala ("pandas como convertir texto a fecha"): buscar es parte del trabajo de un analista.
- Equivocarse es normal. Lee el mensaje de error de abajo hacia arriba: la última línea dice qué pasó.
- Si una celda que modifica datos se ejecuta dos veces, puede dar error. Vuelve a ejecutar desde el paso donde lees los archivos.

### Paso 1 — Importar las librerías

Las mismas de siempre: pandas, numpy y plotly.

In [ ]:
# PASO 1 — IMPORTAR LAS LIBRERÍAS
# 1. Importa pandas (pd), numpy (np), plotly.express (px) y plotly.graph_objects (go)
# 2. Importa make_subplots desde plotly.subplots

# ✍️ Tu código aquí:


### Paso 2 — Importar las bases (ya como fechas)

Con `parse_dates`, pandas convierte las columnas de fecha al leer el archivo.

In [ ]:
# PASO 2 — IMPORTAR LAS BASES (YA COMO FECHAS)
# 1. Lee "entregas.csv" convirtiendo data_hora_pedido y data_hora_entrega en fechas:
#    pd.read_csv("entregas.csv", parse_dates=["data_hora_pedido", "data_hora_entrega"])
# 2. Lee "clima.csv" convirtiendo la columna data
# 3. Verifica con .info() que las columnas de fecha son datetime64
# 4. Mira las primeras filas de cada tabla

# ✍️ Tu código aquí:


### Paso 3 — Crear variables de tiempo

De una fecha-hora se sacan muchas variables útiles: duración, hora, día de la semana, mes.

In [ ]:
# PASO 3 — CREAR VARIABLES DE TIEMPO
# 1. tempo_min = (data_hora_entrega - data_hora_pedido) en minutos
#    💡 Pista: la resta da un timedelta → .dt.total_seconds() / 60
# 2. data = solo la fecha del pedido, sin la hora → .dt.normalize()
# 3. hora = .dt.hour
# 4. dia_semana: .dt.dayofweek da 0=lunes ... 6=domingo → conviértelo en nombres con .map()
#    usa la lista dias_pt = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
# 5. mes = .dt.month
# 6. atrasado = tempo_min > 45 (True/False)
# 7. Rellena los vacíos de cupom con el texto "Sem cupom"

# ✍️ Tu código aquí:


### Paso 4 — Unir con el clima

Para saber si llovió el día de cada pedido, une las dos tablas por la fecha.

In [ ]:
# PASO 4 — UNIR CON EL CLIMA
# 1. En clima, crea dia_chuvoso = chuva_mm >= 5 (lluvia considerable)
# 2. Une entregas con clima por la columna data (how="left") → df
#    ⚠️ Para que funcione, las dos columnas "data" deben ser del mismo tipo (datetime, sin hora)
# 3. Cuenta cuántos pedidos hubo en días con y sin lluvia

# ✍️ Tu código aquí:


### Paso 5 — KPIs y SLA por barrio

Los números de la portada del dashboard y la tabla que dice **dónde** falla la operación.

In [ ]:
# PASO 5 — KPIS Y SLA POR BARRIO
# 1. Imprime: total de entregas, tiempo medio, tiempo mediano, % dentro del SLA y nota media
# 2. Agrupa por bairro y calcula:
#    - entregas (count), tempo_medio (mean), p90 = percentil 90 del tiempo,
#      pct_atraso (media de atrasado), distancia_media
#    💡 Pista para el p90: ("tempo_min", lambda s: s.quantile(0.9))
# 3. Ordena por pct_atraso de mayor a menor → sla_bairro
# 🤔 ¿El barrio con más atrasos es también el más lejano? ¿Es solo distancia?

# ✍️ Tu código aquí:


### Paso 6 — Mapa de calor de la demanda

Una **tabla dinámica** (*pivot table*) cruza día de la semana × hora. Es el gráfico que usa Operaciones para planificar turnos.

In [ ]:
# PASO 6 — MAPA DE CALOR DE LA DEMANDA
# 1. Cuenta pedidos por dia_semana (filas) y hora (columnas):
#    df.pivot_table(index="dia_semana", columns="hora", values="pedido_id", aggfunc="count", fill_value=0)
# 2. Divide por la cantidad de semanas (días únicos / 7) → pedidos PROMEDIO por franja
# 3. Ordena las filas de lunes a domingo con .reindex(dias_pt)
# 4. Dibújalo con px.imshow(..., color_continuous_scale=[...], aspect="auto")
#    💡 Para cantidades usa UNA sola tonalidad: de azul claro a azul oscuro

# ✍️ Tu código aquí:


### Paso 7 — Serie temporal: pedidos por día

Los datos diarios suben y bajan mucho de un día a otro. La **media móvil de 7 días** suaviza la serie y deja ver la tendencia.

In [ ]:
# PASO 7 — SERIE TEMPORAL: PEDIDOS POR DÍA
# 1. Agrupa por data: cantidad de pedidos y tiempo medio → diario
# 2. Crea media_movel_7d = diario["pedidos"].rolling(7).mean()
# 3. Total por semana con resample: df.set_index("data_hora_pedido").resample("W")["pedido_id"].count()
# 4. Pedidos por mes y variación % con .pct_change()
# 5. Gráfico: línea gris fina para los pedidos diarios + línea azul más gruesa para la media móvil

# ✍️ Tu código aquí:


### Paso 8 — ¿La lluvia cambia todo?

Compara los días con y sin lluvia: volumen de pedidos, tiempo medio y % de atrasos.

In [ ]:
# PASO 8 — ¿LA LLUVIA CAMBIA TODO?
# 1. Agrega a "diario" las columnas chuva_mm y dia_chuvoso de clima
#    💡 Pista: diario.join(clima.set_index("data")[["chuva_mm", "dia_chuvoso"]])
# 2. Agrupa diario por dia_chuvoso: cantidad de días, pedidos medios por día y tiempo medio
# 3. Con df, calcula el % de atrasos con y sin lluvia
# 4. ¿Cuántos % más de pedidos hay un día de lluvia? ¿Cuántos minutos más tarda la entrega?

# ✍️ Tu código aquí:


### Paso 9 — Demora × nota del cliente

Con `pd.cut` creas **franjas** (rangos) de tiempo y comparas la nota media de cada franja.

In [ ]:
# PASO 9 — DEMORA × NOTA DEL CLIENTE
# 1. Crea faixa_tempo con pd.cut(df["tempo_min"], bins=[0, 30, 45, 60, 90, np.inf],
#    labels=["≤30 min", "30–45", "45–60", "60–90", ">90"])
# 2. Agrupa por faixa_tempo (usa observed=True): cantidad de entregas, nota media
#    y % de notas 1 o 2 (pista: lambda s: s.dropna().le(2).mean())
# 3. ¿A partir de qué franja la nota cae fuerte?

# ✍️ Tu código aquí:


### Paso 10 — Cupones

¿Quién usa cupón gasta más? ¿Qué cupón regala más tarifa de envío?

In [ ]:
# PASO 10 — CUPONES
# 1. Agrupa por cupom: cantidad de pedidos, ticket medio (valor_pedido),
#    tarifa media (taxa_entrega) y nota media → cupons
# 2. Ordena por cantidad de pedidos
# 3. Compara el ticket medio con cupón vs. "Sem cupom"

# ✍️ Tu código aquí:


### Paso 11 — Dashboard operativo

Un panel para la reunión semanal: KPIs arriba, después la demanda en el tiempo, el mapa de calor y los detalles. Se guarda en HTML interactivo.

In [ ]:
# PASO 11 — DASHBOARD OPERATIVO
# 1. make_subplots con 5 filas x 4 columnas:
#    fila 1: 4 indicadores (Entregas, Tiempo medio, % en el SLA, Nota media)
#    fila 2: pedidos por día + media móvil (colspan=4)
#    fila 3: mapa de calor de la demanda (colspan=4) → go.Heatmap
#    fila 4: % de atraso por barrio | tiempo medio por hora con y sin lluvia
#    fila 5: nota media por franja de tiempo | ticket medio por cupón
# 2. Las barras de una sola serie van en un solo color; las dos líneas de lluvia en azul y naranja con leyenda
# 3. Guarda con fig.write_html("dashboard_rapidinho.html")

# ✍️ Tu código aquí:


---
# 🔑 Parte B — Solución comentada

**Intenta resolver la Parte A antes de mirar esto.** Después compara: no hay una única forma correcta,
pero fíjate en las funciones que usé y en los comentarios que explican el porqué.

> Ejecuta la Parte B **en orden, desde el Paso 1**: vuelve a leer los archivos originales, así que no depende de lo que hiciste en la Parte A.

### ✅ Paso 1 — Importar las librerías

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#c3c2b7"  # colores fijos del dashboard

### ✅ Paso 2 — Importar las bases (ya como fechas)

In [ ]:
entregas = pd.read_csv("entregas.csv", parse_dates=["data_hora_pedido", "data_hora_entrega"])
clima = pd.read_csv("clima.csv", parse_dates=["data"])

entregas.info()
display(entregas.head())
display(clima.head())

### ✅ Paso 3 — Crear variables de tiempo

In [ ]:
entregas["tempo_min"] = (entregas["data_hora_entrega"] - entregas["data_hora_pedido"]).dt.total_seconds() / 60
entregas["data"] = entregas["data_hora_pedido"].dt.normalize()  # fecha a las 00:00
entregas["hora"] = entregas["data_hora_pedido"].dt.hour

dias_pt = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
entregas["dia_semana"] = entregas["data_hora_pedido"].dt.dayofweek.map(dict(enumerate(dias_pt)))
entregas["mes"] = entregas["data_hora_pedido"].dt.month
entregas["atrasado"] = entregas["tempo_min"] > 45  # SLA = 45 minutos
entregas["cupom"] = entregas["cupom"].fillna("Sem cupom")

display(entregas[["data_hora_pedido", "tempo_min", "data", "hora", "dia_semana", "atrasado"]].head())

### ✅ Paso 4 — Unir con el clima

In [ ]:
clima["dia_chuvoso"] = clima["chuva_mm"] >= 5
df = entregas.merge(clima, on="data", how="left")

print("Filas:", len(entregas), "→", len(df))
print(df["dia_chuvoso"].value_counts())

### ✅ Paso 5 — KPIs y SLA por barrio

In [ ]:
print(f"Entregas:              {len(df)}")
print(f"Tiempo medio:          {df['tempo_min'].mean():.1f} min (mediana {df['tempo_min'].median():.1f})")
print(f"Dentro del SLA (≤45):  {1 - df['atrasado'].mean():.1%}")
print(f"Nota media:            {df['avaliacao'].mean():.2f}")

sla_bairro = (df.groupby("bairro")
              .agg(entregas=("pedido_id", "count"),
                   tempo_medio=("tempo_min", "mean"),
                   p90=("tempo_min", lambda s: s.quantile(0.9)),
                   pct_atraso=("atrasado", "mean"),
                   distancia_media=("distancia_km", "mean"))
              .sort_values("pct_atraso", ascending=False))
display(sla_bairro.round(2))

💡 **¿Por qué el percentil 90?** La media esconde a los clientes que más sufren. "El 90% de las entregas llega en menos de X minutos" describe mucho mejor la experiencia. Fíjate en el **Centro**: es cerca, pero tiene atrasos por tránsito. Entonces no es solo una cuestión de distancia.

### ✅ Paso 6 — Mapa de calor de la demanda

In [ ]:
n_semanas = df["data"].nunique() / 7
demanda = (df.pivot_table(index="dia_semana", columns="hora", values="pedido_id",
                          aggfunc="count", fill_value=0)
           .reindex(index=dias_pt, columns=range(24), fill_value=0) / n_semanas).round(1)

fig = px.imshow(demanda, aspect="auto",
                color_continuous_scale=["#cde2fb", "#3987e5", "#0d366b"],
                labels=dict(x="Hora del día", y="", color="Pedidos"),
                title="¿Cuándo piden los clientes? Pedidos promedio por franja")
fig.update_traces(xgap=2, ygap=2, hovertemplate="%{y} %{x}h: %{z:.1f} pedidos<extra></extra>")
fig.update_layout(template="plotly_white", xaxis_dtick=2)
fig.show()

### ✅ Paso 7 — Serie temporal: pedidos por día

In [ ]:
diario = df.groupby("data").agg(pedidos=("pedido_id", "count"), tempo_medio=("tempo_min", "mean"))
diario["media_movel_7d"] = diario["pedidos"].rolling(7).mean()

semanal = df.set_index("data_hora_pedido").resample("W")["pedido_id"].count()
print("Primeras semanas:\n", semanal.head(3))

mensal = df.groupby("mes")["pedido_id"].count()
display(pd.DataFrame({"pedidos": mensal, "variacao_%": (mensal.pct_change() * 100).round(1)}))

fig = go.Figure()
fig.add_trace(go.Scatter(x=diario.index, y=diario["pedidos"], name="Pedidos por día",
                         line=dict(color=CINZA, width=1)))
fig.add_trace(go.Scatter(x=diario.index, y=diario["media_movel_7d"], name="Media móvil 7 días",
                         line=dict(color=AZUL, width=2.5)))
fig.update_layout(title="Pedidos por día", template="plotly_white", hovermode="x unified")
fig.show()

💡 **Ojo con comparar meses:** febrero parece caer un 17%, pero tiene 28 días. Para comparar meses de largo distinto, usa **pedidos por día** (`mensal / días del mes`). La primera semana también sale baja porque está incompleta (el 1/1/2026 fue jueves).

### ✅ Paso 8 — ¿La lluvia cambia todo?

In [ ]:
diario = diario.join(clima.set_index("data")[["chuva_mm", "dia_chuvoso"]])

efeito = diario.groupby("dia_chuvoso").agg(dias=("pedidos", "size"),
                                           pedidos_por_dia=("pedidos", "mean"),
                                           tempo_medio=("tempo_medio", "mean"))
efeito.index = efeito.index.map({False: "Sin lluvia", True: "Con lluvia"})
display(efeito.round(1))

atraso_chuva = df.groupby("dia_chuvoso")["atrasado"].mean()
mais_pedidos = efeito.loc["Con lluvia", "pedidos_por_dia"] / efeito.loc["Sin lluvia", "pedidos_por_dia"] - 1
mais_minutos = efeito.loc["Con lluvia", "tempo_medio"] - efeito.loc["Sin lluvia", "tempo_medio"]
print(f"Días con lluvia: {mais_pedidos:+.0%} pedidos y {mais_minutos:+.1f} min por entrega")
print(f"Atrasos: {atraso_chuva[False]:.1%} sin lluvia → {atraso_chuva[True]:.1%} con lluvia")

💡 **Insight de negocio:** con lluvia hay **más** pedidos y la entrega **tarda más**. Las dos cosas juntas revientan el SLA. Una acción posible: un "modo lluvia", con bono para repartidores y tiempo prometido más largo en la app, activado cuando el pronóstico supere 5 mm.

### ✅ Paso 9 — Demora × nota del cliente

In [ ]:
df["faixa_tempo"] = pd.cut(df["tempo_min"], bins=[0, 30, 45, 60, 90, np.inf],
                           labels=["≤30 min", "30–45", "45–60", "60–90", ">90"])
nota_faixa = df.groupby("faixa_tempo", observed=True).agg(
    entregas=("pedido_id", "count"),
    nota_media=("avaliacao", "mean"),
    pct_nota_1_2=("avaliacao", lambda s: s.dropna().le(2).mean()),
)
display(nota_faixa.round(2))

### ✅ Paso 10 — Cupones

In [ ]:
cupons = (df.groupby("cupom")
          .agg(pedidos=("pedido_id", "count"), ticket_medio=("valor_pedido", "mean"),
               taxa_media=("taxa_entrega", "mean"), nota_media=("avaliacao", "mean"))
          .sort_values("pedidos", ascending=False))
display(cupons.round(2))

### ✅ Paso 11 — Dashboard operativo

In [ ]:
fig = make_subplots(
    rows=5, cols=4,
    specs=[[{"type": "indicator"}] * 4,
           [{"colspan": 4}, None, None, None],
           [{"colspan": 4}, None, None, None],
           [{"colspan": 2}, None, {"colspan": 2}, None],
           [{"colspan": 2}, None, {"colspan": 2}, None]],
    row_heights=[0.07, 0.23, 0.25, 0.225, 0.225],
    vertical_spacing=0.07, horizontal_spacing=0.14,
    subplot_titles=["", "", "", "",
                    "Pedidos por día",
                    "Demanda: pedidos promedio por día de la semana y hora",
                    "% de entregas fuera del SLA (>45 min), por barrio",
                    "Tiempo medio de entrega por hora (min)",
                    "Nota media según el tiempo de entrega",
                    "Ticket medio por cupón (R$)"],
)

# Fila 1 — KPIs
kpis = [("Entregas", len(df), "", ",.0f"),
        ("Tiempo medio", df["tempo_min"].mean(), " min", ".1f"),
        ("Dentro del SLA", 100 * (1 - df["atrasado"].mean()), "%", ".1f"),
        ("Nota media", df["avaliacao"].mean(), " / 5", ".2f")]
for i, (titulo, valor, sufixo, formato) in enumerate(kpis, start=1):
    fig.add_trace(go.Indicator(mode="number", value=valor, title={"text": titulo, "font": {"size": 15}},
                               number={"suffix": sufixo, "valueformat": formato, "font": {"size": 34}}),
                  row=1, col=i)

# Fila 2 — serie diaria
fig.add_trace(go.Scatter(x=diario.index, y=diario["pedidos"], name="Pedidos por día",
                         line=dict(color=CINZA, width=1), legend="legend"), row=2, col=1)
fig.add_trace(go.Scatter(x=diario.index, y=diario["media_movel_7d"], name="Media móvil 7 días",
                         line=dict(color=AZUL, width=2.5), legend="legend"), row=2, col=1)

# Fila 3 — mapa de calor (barra de color alineada con su fila)
y3 = fig.get_subplot(3, 1).yaxis.domain
fig.add_trace(go.Heatmap(z=demanda.values, x=demanda.columns, y=demanda.index, xgap=2, ygap=2,
                         colorscale=[[0, "#cde2fb"], [0.5, "#3987e5"], [1, "#0d366b"]],
                         colorbar=dict(title="Pedidos", thickness=12, len=y3[1] - y3[0],
                                       y=(y3[0] + y3[1]) / 2, yanchor="middle"),
                         hovertemplate="%{y} %{x}h: %{z:.1f} pedidos<extra></extra>"), row=3, col=1)

# Fila 4 — barrios | lluvia por hora (10h a 23h, donde hay volumen)
orden = sla_bairro.sort_values("pct_atraso")
fig.add_trace(go.Bar(x=100 * orden["pct_atraso"], y=orden.index, orientation="h", marker_color=AZUL,
                     showlegend=False, hovertemplate="%{y}: %{x:.1f}% fuera del SLA<extra></extra>"),
              row=4, col=1)
tempo_hora = df.groupby(["hora", "dia_chuvoso"])["tempo_min"].mean().unstack().loc[10:23]
fig.add_trace(go.Scatter(x=tempo_hora.index, y=tempo_hora[False], name="Sin lluvia",
                         line=dict(color=AZUL, width=2), legend="legend2"), row=4, col=3)
fig.add_trace(go.Scatter(x=tempo_hora.index, y=tempo_hora[True], name="Con lluvia",
                         line=dict(color=LARANJA, width=2), legend="legend2"), row=4, col=3)

# Fila 5 — nota por franja | cupones
fig.add_trace(go.Bar(x=nota_faixa.index.astype(str), y=nota_faixa["nota_media"], marker_color=AZUL,
                     showlegend=False, hovertemplate="%{x}: nota %{y:.2f}<extra></extra>"), row=5, col=1)
cup = cupons.sort_values("ticket_medio")
fig.add_trace(go.Bar(x=cup["ticket_medio"], y=cup.index, orientation="h", marker_color=AZUL,
                     showlegend=False, hovertemplate="%{y}: R$ %{x:,.2f}<extra></extra>"), row=5, col=3)

# Diseño general
y2 = fig.get_subplot(2, 1).yaxis.domain
x4, y4 = fig.get_subplot(4, 3).xaxis.domain, fig.get_subplot(4, 3).yaxis.domain
fig.update_traces(marker_cornerradius=4, selector=dict(type="bar"))
fig.update_yaxes(range=[0, 5], row=5, col=1)
fig.update_yaxes(range=[22, 52], row=4, col=3)  # deja espacio abajo para la leyenda
fig.update_xaxes(dtick=2, row=3, col=1)
fig.update_layout(
    title=dict(text="<b>Rapidinho</b> · Dashboard operativo, 1er semestre 2026", x=0.01, font=dict(size=22)),
    template="plotly_white", height=1750, separators=",.",
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=13, color="#0b0b0b"),
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", bargap=0.35,
    margin=dict(t=100, l=40, r=30, b=40), hovermode="closest",
    legend=dict(orientation="h", x=1, xanchor="right", y=y2[1] + 0.012, yanchor="bottom"),
    legend2=dict(orientation="h", x=x4[0] + 0.01, xanchor="left", y=y4[0] + 0.005, yanchor="bottom",
                 bgcolor="rgba(252,252,251,0.85)"),
)
fig.update_xaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7", zeroline=False)
fig.update_yaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7", zeroline=False)
fig.show()

fig.write_html("dashboard_rapidinho.html", include_plotlyjs="cdn")
print("✅ Dashboard guardado: dashboard_rapidinho.html")

---
## 🎤 Lo que le dirías a Camila
1. **El problema tiene hora y clima:** los picos (12–13 h y 19–21 h) sumados a la lluvia explican la mayor parte de los atrasos.
2. **Barrios:** Itaquera es el peor por distancia. El Centro, aunque es el barrio más cercano, está entre los que más atrasan: ahí el problema es el tránsito. Son causas distintas con soluciones distintas.
3. **La demora cuesta reputación:** a partir de los 45 minutos la nota cae fuerte. Cumplir el SLA protege la nota del app.
4. **Acciones:** reforzar repartidores viernes a domingo de 19 a 21 h, un "modo lluvia" y revisar la zona de Itaquera (¿un restaurante socio más cerca?).

## 🚀 Retos extra
1. Ranking de repartidores: tiempo medio ajustado por distancia (`tempo_min / distancia_km`). Ojo con quien hizo pocas entregas.
2. ¿Hay relación entre `temp_max_c` y los pedidos de **Doces** o **Saudável**?
3. Calcula el tiempo de entrega por km para cada barrio. ¿Dónde está el peor tránsito?
4. Usa `resample("W")` para graficar el % de atrasos semana a semana y marca las semanas con más lluvia.